# 05 - Data Cleaning

## PB-07: Limpieza y transformación de variables

Este notebook documenta la etapa de limpieza del dataset para el Sprint 2.

### Objetivo
Transformar el dataset crudo en una versión intermedia limpia y consistente, lista para continuar con feature engineering y pipeline.

### Alcance de PB-07
- revisión de tipos de datos
- verificación de valores nulos
- tratamiento de filas duplicadas
- análisis de outliers
- decisión sobre encoding
- decisión sobre escalado
- guardado del dataset intermedio en `data/interim/`

# 05 - Data Cleaning

## PB-07: Limpieza y transformación de variables

Este notebook documenta la etapa de limpieza del dataset para el Sprint 2.

### Objetivo
Transformar el dataset crudo en una versión intermedia limpia y consistente, lista para continuar con feature engineering y pipeline.

### Alcance de PB-07
- revisión de tipos de datos
- verificación de valores nulos
- tratamiento de filas duplicadas
- análisis de outliers
- decisión sobre encoding
- decisión sobre escalado
- guardado del dataset intermedio en `data/interim/`

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv("../data/raw/dataset.csv")
df = df_raw.copy()

df.shape

(11055, 31)

## Resumen inicial del dataset

Se parte del dataset crudo explorado en el Sprint 1 y se trabaja sobre una copia para no sobrescribir la fuente original.

In [3]:
display(df.head())
display(df.dtypes)

,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


having_IP_Address              int64
URL_Length                     int64
Shortining_Service             int64
having_At_Symbol               int64
double_slash_redirecting       int64
Prefix_Suffix                  int64
having_Sub_Domain              int64
SSLfinal_State                 int64
Domain_registeration_length    int64
Favicon                        int64
port                           int64
HTTPS_token                    int64
Request_URL                    int64
URL_of_Anchor                  int64
Links_in_tags                  int64
SFH                            int64
Submitting_to_email            int64
Abnormal_URL                   int64
Redirect                       int64
on_mouseover                   int64
RightClick                     int64
popUpWidnow                    int64
Iframe                         int64
age_of_domain                  int64
DNSRecord                      int64
web_traffic                    int64
Page_Rank                      int64
G

## Valores nulos

Se verifica si existen valores nulos en el dataset y se decide si es necesario aplicar imputación.

In [4]:
null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values("null_count", ascending=False)

null_summary

,null_count,null_pct
having_IP_Address,0,0.0
URL_Length,0,0.0
Shortining_Service,0,0.0
having_At_Symbol,0,0.0
double_slash_redirecting,0,0.0
Prefix_Suffix,0,0.0
having_Sub_Domain,0,0.0
SSLfinal_State,0,0.0
Domain_registeration_length,0,0.0
Favicon,0,0.0


## Decisión sobre valores nulos

No se detectan valores nulos en el dataset, por lo que **no se aplica imputación** en esta etapa.

## Filas duplicadas

Se cuantifican las filas duplicadas exactas para decidir si deben eliminarse del dataset intermedio.

In [5]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(5206)

## Decisión sobre filas duplicadas

Se eliminan las filas duplicadas exactas, ya que representan redundancia en el dataset y pueden sesgar etapas posteriores del Sprint 2.

La limpieza se hace sobre una copia del dataset y el resultado será guardado como dataset intermedio.

In [6]:
df_clean = df.drop_duplicates().copy()

print("Shape original:", df.shape)
print("Shape sin duplicados:", df_clean.shape)

Shape original: (11055, 31)
Shape sin duplicados: (5849, 31)


## Tipos de datos

Se revisa si hay variables con tipos inconsistentes o si es necesario corregir categorías, fechas o columnas numéricas.

In [7]:
dtype_summary = df_clean.dtypes.value_counts()
dtype_summary

int64    31
Name: count, dtype: int64

## Decisión sobre tipos de datos

Todas las variables se encuentran en formato numérico (`int64`), lo cual es consistente con la codificación del dataset.

No se requiere conversión adicional de fechas ni de variables categóricas en esta etapa.

## Outliers

Se realiza una detección exploratoria de outliers usando IQR sobre las variables numéricas.

In [8]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

outlier_counts = {}

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_counts[col] = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()

outlier_summary = pd.DataFrame.from_dict(
    outlier_counts, orient="index", columns=["outlier_count"]
).sort_values("outlier_count", ascending=False)

outlier_summary

,outlier_count
popUpWidnow,1275
Submitting_to_email,1243
Favicon,1223
having_At_Symbol,1203
URL_Length,1171
HTTPS_token,1013
Google_Index,989
port,943
Abnormal_URL,941
Statistical_report,886


## Decisión sobre outliers

Aunque el método IQR detecta observaciones fuera del rango esperado, este dataset está compuesto en gran parte por variables discretas codificadas en valores como `-1`, `0` y `1`.

Por ello, los “outliers” detectados no se interpretan como errores numéricos clásicos y **no se aplicará winsorizing ni eliminación de registros por outliers** en esta etapa.

## Decisión sobre encoding

No se aplica encoding adicional en PB-07 porque las variables del dataset ya se encuentran en formato numérico codificado.

## Decisión sobre escalado

No se aplica escalado directamente sobre el dataset completo en esta etapa.

El escalado se diferirá al pipeline posterior del Sprint 2, ya que técnicas como `StandardScaler`, `MinMaxScaler` o `RobustScaler` aprenden parámetros del dato y deben ajustarse solo sobre el conjunto de entrenamiento para evitar data leakage.

## Verificación final del dataset limpio

Se valida el estado del dataset intermedio luego de las decisiones de limpieza.

In [9]:
print("Shape final:", df_clean.shape)
print("Nulos finales:", int(df_clean.isnull().sum().sum()))
print("Duplicados finales:", int(df_clean.duplicated().sum()))
print("Tipos de datos:")
display(df_clean.dtypes.value_counts())

Shape final: (5849, 31)
Nulos finales: 0
Duplicados finales: 0
Tipos de datos:


int64    31
Name: count, dtype: int64

## Guardado del dataset intermedio

Se exporta el dataset limpio a `data/interim/` para su uso en la siguiente etapa del Sprint 2.

In [10]:
output_path = "../data/interim/clean_dataset.csv"
df_clean.to_csv(output_path, index=False)

print(f"Dataset intermedio guardado en: {output_path}")

Dataset intermedio guardado en: ../data/interim/clean_dataset.csv


## Resumen de PB-07

En esta notebook se realizó la limpieza base del dataset:

- no se detectaron valores nulos
- se eliminaron filas duplicadas
- se verificó consistencia de tipos de datos
- se analizó la presencia de outliers
- se documentó por qué no se aplicó encoding adicional
- se documentó por qué el escalado se difiere al pipeline posterior
- se guardó un dataset intermedio en `data/interim/clean_dataset.csv`

Esta notebook corresponde al backlog item **PB-07: Limpieza y transformación de variables**.